# P29 — Árbol de pensamientos: resolución deliberada de problemas con modelos de lenguaje grandes

## 1. Título y paper

**Paper:** *Tree of Thoughts: Deliberate Problem Solving with Large Language Models*  
**Autoría:** Shunyu Yao, Dian Yu, Jeffrey Zhao, Izhak Shafran, Thomas L. Griffiths, Yuan Cao, Karthik Narasimhan  
**Año y venue:** 2023 · arXiv:2305.10601 · NeurIPS 2023  
**Nivel:** L3 · **Motor:** `tot`  
**Ficha completa:** [`P29_tree_of_thoughts`](../../papers/foundational/P29_tree_of_thoughts/README.md)

**Hito:** Devuelve la búsqueda clásica al razonamiento: explorar varias ramas, evaluarlas y poder retroceder.

- [arXiv:2305.10601](https://arxiv.org/abs/2305.10601)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Una cadena de pensamiento decide de izquierda a derecha y sin vuelta atrás: un paso localmente razonable y globalmente equivocado condena toda la solución.
2. Ejecutar una implementación mínima de la propuesta: Tratar los pasos de razonamiento como nodos de un árbol, hacer que el modelo evalúe estados parciales y aplicar búsqueda con poda y retroceso.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P28
- P27
- búsqueda en espacios de estados


## 4. Intuición

Una cadena de pensamiento es como escribir a bolígrafo: si el tercer paso está mal, sigues adelante con él. Un árbol es escribir a lápiz con varias hojas: exploras, comparas y borras.


## 5. Concepto mínimo

```text
Cadena :  s₀ → s₁ → s₂ → s₃            una rama, sin vuelta atrás
Árbol  :  s₀ → {s₁ᵃ, s₁ᵇ, s₁ᶜ} → …     varias ramas, con evaluación y poda
```

Tres piezas necesarias: **generar** candidatos, **evaluar** estados parciales (el propio modelo juzga «esto promete / esto no lleva a nada») y una **estrategia de búsqueda** (anchura, profundidad, poda).


## 6. Código explicado

El motor compara una cadena lineal con una búsqueda en árbol con poda sobre el mismo espacio.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('tot', seed=7)['result']
print('profundidad:', r['profundidad'], '· ramas por paso:', r['ramas_por_paso'], '\n')
show(r['cadena_lineal'])
show(r['busqueda_en_arbol'])
print('\ncoste relativo:', r['coste_relativo'], 'x')

## 7. Predicción antes de ejecutar

1. ¿Cuántos nodos evalúa una cadena de profundidad 3 con 3 ramas? ¿Y un árbol con anchura 3?
2. ¿Qué gana el árbol a cambio de ese coste?
3. Si el evaluador fuera aleatorio, ¿serviría de algo explorar?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for anchura in (1, 2, 3, 5):
    nodos = sum(min(3 ** (d + 1), anchura * 3) for d in range(3))
    print(f'anchura {anchura} → ~{nodos:>2} nodos evaluados '
          f"({'equivale a la cadena lineal' if anchura == 1 else 'mantiene alternativas vivas'})")

## 9. Salida interpretable

Con anchura 1, el árbol **es** la cadena lineal: ese es el caso límite. Cada unidad de anchura multiplica el coste y compra la posibilidad de recuperarse de un mal paso. El compromiso es explícito y se puede presupuestar.


## 10. Comentario pedagógico

La pieza frágil es el **evaluador**. Si el modelo no sabe juzgar estados parciales, el árbol solo multiplica el gasto. Por eso ToT funciona bien en problemas donde el progreso parcial es verificable (Game of 24, crucigramas) y peor donde no lo es.


## 11. Error o anti-patrón deliberado

Anti-patrón: aumentar la anchura para «buscar mejor», sin comprobar la calidad del evaluador.


In [ ]:
import random
rng = random.Random(0)
for calidad in (0.5, 0.7, 0.95):
    aciertos = sum(1 for _ in range(1000) if rng.random() < calidad)
    print(f'evaluador con {calidad:.0%} de acierto → poda correcta {aciertos/10:.1f}% de las veces')
print('\ncon un evaluador al 50% la poda es una moneda: gastas 3x y no ganas nada')

## 12. Corrección

La corrección es medir el evaluador **antes** de pagar la búsqueda:


In [ ]:
protocolo = {
    'paso_1': 'medir la calidad del evaluador sobre estados parciales con solución conocida',
    'paso_2': 'si acierta poco, mejorar el evaluador ANTES de ampliar la búsqueda',
    'paso_3': 'fijar presupuesto de nodos y reportarlo junto con la exactitud',
}
show(protocolo)

## 13. Desafío guiado

Comprueba el caso límite: con anchura 1 el árbol debe comportarse exactamente como la cadena.


In [ ]:
r = run_paper_lab('tot', seed=7)['result']
print('cadena lineal   :', r['cadena_lineal']['nodos_evaluados'], 'nodos')
print('árbol anchura 3 :', r['busqueda_en_arbol']['nodos_evaluados'], 'nodos')
print('frontera final  :', r['busqueda_en_arbol']['frontera_final'], 'hipótesis vivas')

## 14. Desafío autónomo

Implementa ToT sobre el juego de las 24 con un modelo abierto: genera candidatos, haz que el modelo clasifique cada estado parcial como «seguro / quizá / imposible», y compara la tasa de éxito frente a cadena simple. Reporta también el número de llamadas al modelo.


## 15. Evidencia de aprendizaje

Guarda la comparación de nodos evaluados, el caso límite de anchura 1 y tu protocolo para medir el evaluador antes de pagar la búsqueda.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P29_tree_of_thoughts/README.md) · evaluación formal: [`assessments/papers/P29_tree_of_thoughts.md`](../../assessments/papers/P29_tree_of_thoughts.md)


## 16. Cierre

Deliberar mejor dentro de un intento. Falta aprender **entre** intentos: que fallar una vez sirva para la siguiente.


## 17. Conexión con el siguiente hito

- P22

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
